In [2]:
library("lme4")
library("margins")
library("stargazer")
library("emmeans")
library("ggeffects")
library("broom")
library("broom.mixed")
library("MASS")
library("pscl")
library("fixest")
library("marginaleffects")
library("modelsummary")
library("glmmTMB")
library("dplyr")

In [3]:
main_path <- "/home/20250114zmz_kd/"
data <- read.csv(paste0(main_path, "UseBSForNot/MergeInsthData_1211.csv"))
dim(data)

[1] 3227892      78

In [4]:
print(names(data))

 [1] "X"                                           
 [2] "work_id"                                     
 [3] "fac_pub"                                     
 [4] "year"                                        
 [5] "paper_type"                                  
 [6] "paper_language"                              
 [7] "nwbin"                                       
 [8] "new_word_reuse_times"                        
 [9] "npbin"                                       
[10] "new_phrase_reuse_times"                      
[11] "novelbin"                                    
[12] "rao_stirling"                                
[13] "author_id"                                   
[14] "author_position"                             
[15] "institution_id"                              
[16] "country_code"                                
[17] "country"                                     
[18] "num_author_log"                              
[19] "num_inst_log"                                
[20] "num_co

In [5]:
colSums(is.na(data))

X 
                                           0 
                                     work_id 
                                           0 
                                     fac_pub 
                                           0 
                                        year 
                                           0 
                                  paper_type 
                                           0 
                              paper_language 
                                           0 
                                       nwbin 
                                           0 
                        new_word_reuse_times 
                                           0 
                                       npbin 
                                           0 
                      new_phrase_reuse_times 
                                           0 
                                    novelbin 
                                           0 
                                rao_stirling 
                                      303430 
                                   author_id 
                                           0 
                             author_position 
                                           0 
                              institution_id 
                                           0 
                                country_code 
                                           0 
                                     country 
                                           0 
                              num_author_log 
                                           0 
                                num_inst_log 
                                           0 
                             num_country_log 
                                           0 
                           num_reference_log 
                                           0 
                             timescited5_log 
                                      107147 
                            timescited10_log 
                                      163886 
                           timescitedall_log 
                                           0 
                               ab_length_log 
                                           0 
                     leadership_global_class 
                                           0 
                         mean_career_age_log 
                                           0 
                            inst_h_index_log 
                                           6 
                                   source_id 
                                           0 
                          source_h_index_log 
                                           0 
                                 source_type 
                                           0 
                                 core_source 
                                           0 
        Agricultural.and.Biological.Sciences 
                                           0 
                         Arts.and.Humanities 
                                           0 
Biochemistry..Genetics.and.Molecular.Biology 
                                           0 
         Business..Management.and.Accounting 
                                           0 
                        Chemical.Engineering 
                                           0 
                                   Chemistry 
                                           0 
                            Computer.Science 
                                           0 
                           Decision.Sciences 
                                           0 
                                   Dentistry 
                                           0 
                Earth.and.Planetary.Sciences 
                                           0 
         Economics..Econometrics.and.Finance 
                                           0 
                                      Energy 
                                         

In [6]:
# 把所有无限值替换成 NA
data[sapply(data, is.infinite)] <- NA

In [7]:
# data <- data[!is.na(data$Atyp_10pct_Z), ]
# dim(data)
data <- data[data$num_reference >=5, ]
dim(data)

[1] 2768162      78

In [8]:
# data <- data[data$ab_length > 0, ]
# dim(data)
data <- data[data$year >= 1950, ]
dim(data)

[1] 2768162      78

In [9]:
# 找出所有包含无限值的行和列
inf_mask <- sapply(data, function(col) is.infinite(col))
rows_with_inf <- apply(inf_mask, 1, any)  # 哪些行至少有一个Inf
cols_with_inf <- colnames(data)[apply(inf_mask, 2, any)]  # 哪些列有Inf

# 打印包含无限值的行数和列名
cat("包含无限值的行数:", sum(rows_with_inf), "\n")
cat("包含无限值的列名:", paste(cols_with_inf, collapse = ", "), "\n")

# 查看这些行具体内容
data_filt_with_inf <- data[rows_with_inf, c(cols_with_inf), drop=FALSE]
print(data_filt_with_inf)

包含无限值的行数: 0 
包含无限值的列名:  
data frame with 0 columns and 0 rows


In [10]:
data$leadership_global_class <- factor(data$leadership_global_class)
data <- within(data, leadership_global_class <- relevel(leadership_global_class, ref = 'shared'))
data$fac_pub <- factor(data$fac_pub)
data <- within(data, fac_pub <- relevel(fac_pub, ref = 'NonBSF'))
data$core_source <- factor(data$core_source)
data <- within(data, core_source <- relevel(core_source, ref = 'NonCore'))
data$year <- as.factor(data$year)

In [11]:
# variable type
paper_level <- "num_author_log + num_inst_log + num_country_log + num_reference_log + leadership_global_class + mean_career_age_log + inst_h_index_log" 
journal_level <- "source_h_index_log + core_source"
disciplines <- "Arts.and.Humanities + Biochemistry..Genetics.and.Molecular.Biology + Business..Management.and.Accounting + Chemical.Engineering + 
 Chemistry + Computer.Science + Decision.Sciences + Dentistry + Earth.and.Planetary.Sciences + 
Economics..Econometrics.and.Finance + Energy + Engineering + Environmental.Science + Health.Professions + 
Immunology.and.Microbiology + Materials.Science + Mathematics + Medicine + Neuroscience + Nursing +
Pharmacology..Toxicology.and.Pharmaceutics + Physics.and.Astronomy + Psychology + Social.Sciences + Veterinary "

In [12]:
fml <- as.formula(
  paste0("rao_stirling ~ fac_pub + ", paper_level, "+", journal_level, "+", disciplines, " | author_id + year")
)

model_ps <- feols(fml, data = data[(data$domain_Physical.Sciences == 1), ], vcov = "iid")
summary(model_ps)

NOTE: 3 observations removed because of NA values (RHS: 3).



OLS estimation, Dep. Var.: rao_stirling
Observations: 2,321,151
Fixed-effects: author_id: 75,686,  year: 76
Standard-errors: IID 
                                              Estimate Std. Error    t value
fac_pubBSF                                   -0.000475   0.000083  -5.748206
num_author_log                               -0.000074   0.000064  -1.153360
num_inst_log                                  0.000826   0.000082  10.031007
num_country_log                              -0.003177   0.000120 -26.557616
num_reference_log                             0.002499   0.000041  61.403041
leadership_global_classallNorth               0.001470   0.000118  12.493404
leadership_global_classallSouth               0.000677   0.000187   3.627642
mean_career_age_log                           0.000533   0.000068   7.876568
inst_h_index_log                             -0.000269   0.000052  -5.147862
source_h_index_log                           -0.001061   0.000031 -34.764285
core_sourceCore        

In [13]:
MEs = ggemmeans(model_ps, terms=c('fac_pub'), typical='median')

In [14]:
MEs

x,predicted,std.error,conf.low,conf.high,group
<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>
NonBSF,0.1368185,0.0004391578,0.1359577,0.1376792,1
BSF,0.1363430,0.0004391761,0.1354822,0.1372037,1


In [15]:
fname = paste0(main_path, "UseBSForNot/predict_rao_1211_ps.csv")
write.csv(MEs, fname, row.names = FALSE)

In [16]:
fml <- as.formula(
  paste0("rao_stirling ~ fac_pub + ", paper_level, "+", journal_level, "+", disciplines, " | author_id + year")
)

model_ls <- feols(fml, data = data[(data$domain_Life.Sciences == 1), ], vcov = "iid")
summary(model_ls)

OLS estimation, Dep. Var.: rao_stirling
Observations: 729,686
Fixed-effects: author_id: 43,960,  year: 75
Standard-errors: IID 
                                                Estimate Std. Error    t value
fac_pubBSF                                    0.01378190   0.000154  89.356033
num_author_log                                0.00112469   0.000112   9.999113
num_inst_log                                  0.00199448   0.000145  13.780334
num_country_log                              -0.00107485   0.000219  -4.915430
num_reference_log                             0.00235265   0.000078  30.133047
leadership_global_classallNorth               0.00137342   0.000269   5.112106
leadership_global_classallSouth              -0.00004073   0.000425  -0.095770
mean_career_age_log                           0.00152598   0.000121  12.614467
inst_h_index_log                              0.00049775   0.000096   5.178392
source_h_index_log                           -0.00000459   0.000054  -0.084858
cor

In [17]:
MEs = ggemmeans(model_ls, terms=c('fac_pub'), typical='median')
MEs

x,predicted,std.error,conf.low,conf.high,group
<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>
NonBSF,0.1480510,0.0008399310,0.1464047,0.1496972,1
BSF,0.1618329,0.0008412773,0.1601840,0.1634817,1


In [18]:
fname = paste0(main_path, "UseBSForNot/predict_rao_1211_ls.csv")
write.csv(MEs, fname, row.names = FALSE)

In [19]:
fml <- as.formula(
  paste0("rao_stirling ~ fac_pub + ", paper_level, "+", journal_level, "+", disciplines, " | author_id + year")
)

model_hs <- feols(fml, data = data[(data$domain_Health.Sciences == 1), ], vcov = "iid")
summary(model_hs)

NOTE: 2 observations removed because of NA values (LHS: 1, RHS: 1).



OLS estimation, Dep. Var.: rao_stirling
Observations: 376,452
Fixed-effects: author_id: 35,489,  year: 75
Standard-errors: IID 
                                              Estimate Std. Error    t value
fac_pubBSF                                    0.018903   0.000255  74.270804
num_author_log                               -0.000162   0.000170  -0.954099
num_inst_log                                  0.002437   0.000211  11.526151
num_country_log                               0.000129   0.000326   0.395635
num_reference_log                             0.002476   0.000121  20.544861
leadership_global_classallNorth               0.000424   0.000410   1.034254
leadership_global_classallSouth               0.000508   0.000634   0.800892
mean_career_age_log                          -0.000011   0.000194  -0.058531
inst_h_index_log                              0.002095   0.000142  14.774501
source_h_index_log                            0.000738   0.000085   8.714286
core_sourceCore          

In [20]:
MEs = ggemmeans(model_hs, terms=c('fac_pub'), typical='median')
MEs

x,predicted,std.error,conf.low,conf.high,group
<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>
NonBSF,0.1456366,0.001300044,0.1430886,0.1481847,1
BSF,0.1645401,0.001305936,0.1619805,0.1670997,1


In [21]:
fname = paste0(main_path, "UseBSForNot/predict_rao_1211_hs.csv")
write.csv(MEs, fname, row.names = FALSE)

In [22]:
fml <- as.formula(
  paste0("rao_stirling ~ fac_pub + ", paper_level, "+", journal_level, "+", disciplines, " | author_id + year")
)

model_ss <- feols(fml, data = data[(data$domain_Social.Sciences == 1), ], vcov = "iid")
summary(model_ss)

NOTE: 1 observation removed because of NA values (RHS: 1).



OLS estimation, Dep. Var.: rao_stirling
Observations: 41,437
Fixed-effects: author_id: 13,137,  year: 74
Standard-errors: IID 
                                              Estimate Std. Error   t value
fac_pubBSF                                    0.007255   0.001476  4.914008
num_author_log                                0.005099   0.000908  5.618134
num_inst_log                                  0.000551   0.001101  0.500926
num_country_log                              -0.003916   0.001544 -2.536575
num_reference_log                             0.002004   0.000494  4.056738
leadership_global_classallNorth               0.003179   0.001544  2.059010
leadership_global_classallSouth               0.000749   0.002148  0.348754
mean_career_age_log                           0.001538   0.000905  1.698959
inst_h_index_log                              0.001357   0.000559  2.425517
source_h_index_log                            0.000569   0.000374  1.521079
core_sourceCore                      

In [23]:
MEs = ggemmeans(model_ss, terms=c('fac_pub'), typical='median')
MEs

x,predicted,std.error,conf.low,conf.high,group
<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>
NonBSF,0.2018839,0.005158999,0.1917720,0.2119958,1
BSF,0.2091385,0.005249077,0.1988501,0.2194270,1


In [24]:
fname = paste0(main_path, "UseBSForNot/predict_rao_1211_ss.csv")
write.csv(MEs, fname, row.names = FALSE)

In [25]:
paper_vars <- c("num_author_log", "num_inst_log", "num_country_log",
                "num_reference_log", "leadership_global_class",
                "mean_career_age_log", "inst_h_index_log" )


journal_vars <- c("source_h_index_log", "core_source")

disciplines <- c("Arts.and.Humanities", "Biochemistry..Genetics.and.Molecular.Biology", "Business..Management.and.Accounting",
                 "Chemical.Engineering", "Chemistry", "Computer.Science", "Decision.Sciences", "Dentistry",
                 "Earth.and.Planetary.Sciences", "Economics..Econometrics.and.Finance", "Energy", "Engineering",
                 "Environmental.Science + Health.Professions", "Immunology.and.Microbiology", "Materials.Science", "Mathematics",
                 "Medicine", "Neuroscience", "Nursing", "Pharmacology..Toxicology.and.Pharmaceutics", "Physics.and.Astronomy",
                 "Psychology", "Social.Sciences", "Veterinary")

In [26]:
model1 <- model_ps
model2 <- model_ls
model3 <- model_hs
model4 <- model_ss

country_level_reg <- etable(
    list(model1, model2, model3, model4),
    keep = c("fac_pub", paper_vars, journal_vars ),
    se = "iid",
    # cluster = ~ paper_year + paper_field,
    tex = TRUE,
    digits = 3
)

# writeLines(country_level_reg, con = "selfpromt_reg_1014.tex")
country_level_reg

\begingroup
\centering
\begin{tabular}{lcccc}
   \tabularnewline \midrule \midrule
   Dependent Variable: & \multicolumn{4}{c}{rao\_stirling}\\
   Model:                              & (1)                     & (2)                     & (3)                     & (4)\\  
   \midrule
   \emph{Variables}\\
   fac\_pubBSF                         & -0.0005$^{***}$         & 0.014$^{***}$           & 0.019$^{***}$           & 0.007$^{***}$\\   
                                       & ($8.27\times 10^{-5}$)  & (0.0002)                & (0.0003)                & (0.001)\\   
   num\_author\_log                    & $-7.42\times 10^{-5}$   & 0.001$^{***}$           & -0.0002                 & 0.005$^{***}$\\   
                                       & ($6.44\times 10^{-5}$)  & (0.0001)                & (0.0002)                & (0.0009)\\   
   num\_inst\_log                      & 0.0008$^{***}$          & 0.002$^{***}$           & 0.002$^{***}$           & 0.0006\\   
                       